> 在线讲义：[第 13 章 · RMSNorm：不减平均值行不行，省在哪](https://wikieden.github.io/beyond-transformer/lessons/13-rmsnorm.html) · 本书《超越 Transformer》

# 第 13 章 · RMSNorm：不减平均值行不行，省在哪

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/13-rmsnorm.html) · 仓库里 `beyond-transformer/notes/lessons/13-rmsnorm.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 两种 norm | 式 4 | `layer_norm(X)`、`rms_norm(X)` | [−1.069, …]、[0.2828, …] |
| 第二步 不变性 | 平移、缩放 | `invariances()` | RMS 平移后变 0.4475 |
| 第三步 运算与参数 | 粗略计数 | `op_count(4096)`、`param_bytes(4096, 65)` | 少 43%；1040 vs 520 KB |

一个向量过 LayerNorm 和 RMSNorm 的数；少算几步、少读一份参数省在哪。
a1 的 rmsnorm（PyTorch 模块）和 layer_norm_ablation 的训练留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
X = np.array([1.0, 2.0, 3.0, 6.0])

**函数 `layer_norm`**：小工具。

In [ ]:
def layer_norm(x, g=None, b=None, eps=1e-5):
    g = np.ones_like(x) if g is None else g
    b = np.zeros_like(x) if b is None else b
    return g * (x - x.mean()) / np.sqrt(x.var() + eps) + b

**函数 `rms_norm`**：a1 式：x / RMS(x) · g，RMS(x) = √(mean(x²) + ε)。

In [ ]:
def rms_norm(x, g=None, eps=1e-5):
    """a1 式：x / RMS(x) · g，RMS(x) = √(mean(x²) + ε)。"""
    g = np.ones_like(x) if g is None else g
    return x / np.sqrt((x * x).mean() + eps) * g

**函数 `invariances`**：平移（每个分量加同一个数）和缩放（整体乘一个数）后，输出变不变。

In [ ]:
def invariances(x=X):
    """平移（每个分量加同一个数）和缩放（整体乘一个数）后，输出变不变。"""
    return {
        "LN 平移 +5": float(np.abs(layer_norm(x + 5) - layer_norm(x)).max()),
        "LN 缩放 ×3": float(np.abs(layer_norm(3 * x) - layer_norm(x)).max()),
        "RMS 平移 +5": float(np.abs(rms_norm(x + 5) - rms_norm(x)).max()),
        "RMS 缩放 ×3": float(np.abs(rms_norm(3 * x) - rms_norm(x)).max()),
    }

**函数 `op_count`**：每个向量的逐元素运算（粗略）：LN 要求均值、减均值、平方、求和、除、乘 γ、加 β；RMS 少了求均值、减均值、加 β。

In [ ]:
def op_count(d: int):
    """每个向量的逐元素运算（粗略）：LN 要求均值、减均值、平方、求和、除、乘 γ、加 β；RMS 少了求均值、减均值、加 β。"""
    ln = {"求均值": d, "减均值": d, "平方求和": 2 * d, "除以标准差": d, "乘 γ": d, "加 β": d}
    rms = {"平方求和": 2 * d, "除以 RMS": d, "乘 γ": d}
    return sum(ln.values()), sum(rms.values())

**函数 `param_bytes`**：每个 norm 的参数：LN 有 γ、β 两份，RMSNorm 只有 γ。

In [ ]:
def param_bytes(d: int, n_norms: int, bytes_per: int = 2):
    """每个 norm 的参数：LN 有 γ、β 两份，RMSNorm 只有 γ。"""
    return 2 * d * n_norms * bytes_per, d * n_norms * bytes_per

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 同一个向量过两种 norm")
print(f"  x = {X.tolist()}：均值 {X.mean()}，标准差 {X.std():.4f}，均方根 {np.sqrt((X * X).mean()):.4f}")
print(f"  LayerNorm：{np.round(layer_norm(X), 4).tolist()}（均值 0、方差 1）")
print(f"  RMSNorm：  {np.round(rms_norm(X), 4).tolist()}（均方根 1，均值不是 0）")
print("\n第二步 对什么不变")
for k, v in invariances().items():
    print(f"  {k}：输出最多变化 {v:.4f}")

In [ ]:
print("\n第三步 省在哪")
ln, rms = op_count(4096)
print(f"  d = 4096 一个向量：LN 约 {ln:,} 次逐元素运算，RMSNorm 约 {rms:,} 次（少 {1 - rms / ln:.0%}）")
a, b = param_bytes(4096, 65)
print(f"  32 层（每层 2 个 norm + 最后 1 个）、bf16：LN 参数 {a / 1024:.0f} KB，RMSNorm {b / 1024:.0f} KB")

## 3. 代码里才看得到的

- **ε 加在根号里面**：RMS 很小时防止除零；a1 规定 ε = 1e-5，和这里一样。
- **实际实现要先升精度**：bf16 下直接平方求和容易溢出或丢精度，a1 提示先把 x 转成 float32 算 RMS，再转回原来的类型；这一步 NumPy 版本里看不到。

## 4. 自己改一改

- 把 x 改成 [−3, −1, 1, 3]（均值 0），两种 norm 的输出是不是一样？
- 把 `op_count` 换成你自己数的版本（比如把“除以”算成乘以倒数），比例变多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> LayerNorm 减均值再除标准差；RMSNorm 不减均值，只除均方根。
> 平移、缩放都不变是 LN；只剩缩放不变是 RMS，缩放才是关键。
> FLOPs 不是运行时间：逐元素操作搬得多、算得少，少一步就少读写一遍。